In [ ]:
# Import Python and ViT modules

import os, time
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms, datasets
import torch.optim as optim
from PIL import Image
import torch.nn.functional as F
from torch.optim.lr_scheduler import OneCycleLR



import geoclip
from geoclip import GeoCLIP

In [ ]:
# Look at the data distribution

file_list = []
image_count = []

for f in os.listdir():
    try:
        file_list.append(int(f))
    except:
        continue
    image_count.append(len(os.listdir(f)))

print(f'There are {len(file_list)} quadrants')

In [ ]:
plt.bar(file_list, image_count)
plt.xlabel('Quad')
plt.ylabel('Amount of images per quad')

In [ ]:
# Display 5 random images

random_quads = random.sample(file_list, 5)

fig, axs = plt.subplots(1, 5, figsize=(20,4))


for quad, ax in zip(random_quads, axs):
    try:
        rand_img = random.choice(os.listdir(str(quad)))
    except IndexError:
        continue
    img = Image.open(str(quad) + '/' + rand_img)

    ax.imshow(img)
    ax.set_title('IMAGE ID: ' + rand_img.strip('.jpg') + ' QUAD #: ' + str(quad), fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Read in data from train.csv, not including test.csv
# may take a few minutes

world_set = pd.read_csv('train.csv')
us_mask = world_set['unique_country'] == 'US'
us_set = world_set[us_mask]

In [ ]:
# Get a list of all the ids in our folders

all_ids = []

for f in os.listdir():
    try:
        images = os.listdir(f)
    except NotADirectoryError:
        continue
    for im in images:
        try:
            all_ids.append(int(im.strip('.jpg')))
        except ValueError:
            pass



In [ ]:
# Get the dataframe for all the us dataset that we actually use (see get_data.ipynb for more info)

used_mask = us_set['id'].isin(all_ids)
us_set_used = us_set[used_mask]
# A dictionary that maps each id to it's coordinates

id_coords = {idd: (lat, long) for idd, lat, long in zip(us_set_used['id'], us_set_used['latitude'], us_set_used['longitude'])}

In [ ]:
# One metric we will use to compare and predict quadrants in the center, which will be
# the average of all the longitude and latitudes in that quadrant
# centers is the full us_set center and centers_used is the one with only the quadrants we use

def findCenter(lats, longs):
    return (np.mean(lats), np.mean(longs))

centers = {}
quad_list = file_list
centers_used = {}

for quad in us_set['quadtree_10_5000'].unique():
    mask = us_set['quadtree_10_5000'] == quad
    quad_set = us_set[mask]
    lats = np.array(quad_set['latitude'])
    longs = np.array(quad_set['longitude'])
    centers[str(quad)] = findCenter(lats, longs)

for quad in us_set_used['quadtree_10_5000'].unique():
    mask = us_set['quadtree_10_5000'] == quad
    quad_set = us_set[mask]
    lats = np.array(quad_set['latitude'])
    longs = np.array(quad_set['longitude'])
    centers_used[str(quad)] = findCenter(lats, longs)

In [ ]:
# Spot check this visually

mask1 = us_set['quadtree_10_5000'] == 516
print(us_set[mask1]['latitude'])
print(us_set[mask1]['longitude'])

# Pretty close to 20.85, -157 roughly hawaii

In [ ]:
# Simple helper function to get distances between center of cells

def getDistance(centers, quad1, quad2):
    quad1_center = centers[quad1]
    quad2_center = centers[quad2]
    return np.linalg.norm(np.array(quad2_center) - np.array(quad1_center))

getDistance(centers, '516', '554')

In [ ]:
# Visualize the data distribution on a map (increase sample size to see individual cells better)

import folium

m = folium.Map(location=[44, -103], zoom_start=12)
colors = ['blue', 'red', 'cyan', 'yellow', 'orange', 'green', 'white', 'black', 'pink']
rand_rows = us_set_used.sample(1000, random_state=60)

for i in range(len(rand_rows)):
    lat = rand_rows.iloc[i]['latitude']
    long = rand_rows.iloc[i]['longitude']
    folium.CircleMarker(
    location=[lat, long],
    radius=10,                 
    color = colors[rand_rows.iloc[i]['quadtree_10_5000'] % len(colors)],          
    fill=True,
    fill_color="cyan"     
    ).add_to(m)

m

In [ ]:
class GeoData:
    def __init__(self, samples, label_matrix, aug=True):
        self.samples = samples # Passed as a tulpe of a item path and a label, ex: samples[0] = (./928/0129393.jpg, 928)
        self.matrix = label_matrix
        normalize = transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],  # Exact same normalization as geoclip
                                        std=[0.26862954, 0.26130258, 0.27577711])
        
        if aug:
            self.transform = transforms.Compose([
                transforms.RandomResizedCrop(224, scale=(0.7, 1)), # Give random parts of images, important to capture real life variations in image size
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05), # Adjust for different brightnesses and image quality. 
                # important for overcast days for example
                transforms.RandomGrayscale(p=0.05), # Occasionally use grayscale, forces model to not become solely dependent on color distributions
                transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2)), # Add random blur to account for image quality
                transforms.ToTensor(), # Required format conversion
                transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)), # Add random erasure to the photo to simulate things being blocked, after tensor
                normalize])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(256), # geo clip image size
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize]) # Required format conversion
            
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cell_id = self.samples[idx]
        img = Image.open(path).convert('RGB') # open the image we want to get
        img = self.transform(img)
        label = self.matrix[cell_id]
        return img, label, cell_id
            
            
                
                
    

In [ ]:
class GeoCLIPClassifier(nn.Module):
    def __init__(self, num_classes=len(file_list), dropout=0.35):
        super().__init__()

        geo_model = geoclip.GeoCLIP()
        self.backbone = geo_model.image_encoder # OpenAI pretrained model that geoclip uses, need to freeze it

        for param in self.backbone.parameters(): # Freeze the backbone
            param.requires_grad = False

        # To get the actual embedding dimensions pass a dummy image
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224) # 1 image, 3 rgb, 224x224
            emb_dim = self.backbone(dummy).shape[-1] 

        self.head = nn.Sequential(nn.LayerNorm(emb_dim), # normalize to the embeeding dimension 
                                  nn.Linear(emb_dim, 1024), # a linear layer
                                  nn.GELU(), # a guassian error linear unit, introduce non linearity between linear functions
                                  nn.Dropout(dropout), # randomly dropout some number of neurons to prevent overfitting
                                  nn.Linear(1024, num_classes)) # the output layer

    def forward(self, x):
        with torch.no_grad(): # don't update params in forward pass
            features = self.backbone(x) 
        logits = self.head(features)
        return logits
                        
                                  
    

In [ ]:
def getGeoLoss(soft_labels, logits): # use KL divergence, cross entropy won't work for soft labeling
    log_probs = F.log_softmax(logits, dim=-1) # softmax and log, kl needs this in log probabilities
    return F.kl_div(log_probs, soft_labels, reduction="batchmean") 

def train(model, train_loader, val_loader, save_path = './geoclip_us_states.pth', epochs=10, lr=3e-4, weight_decay=0.05):
    start_time = time.time()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    optimizer = optim.AdamW(model.head.parameters(), lr=lr, weight_decay=weight_decay)

    best_loss = float('inf')
    
    scheduler = OneCycleLR(optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader), pct_start=0.1, anneal_strategy='cos')

    model.to(device)
    for e in range(epochs): 
        model.train()
        total_loss = 0
        for imgs, soft_labels, cell_ids in train_loader:
            imgs = imgs.to(device)
            soft_labels = soft_labels.to(device)

            logits = model(imgs)
            loss = getGeoLoss(soft_labels, logits)

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.head.parameters(), max_norm=3)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        
        val_metrics = evaluate(model, val_loader, centers_rad, device)
        print(f'EPOCH: {e}')
        print(f'Total loss: {total_loss/len(train_loader):.4f}')
        print(f'Top1: {val_metrics["top1"]:.3f}') # how many times label was in top 1 (first prediction) 
        print(f'Top5: {val_metrics["top5"]:.3f}') # how many times label was in top 5 predictions
        end_epoch_time = time.time()
        dur = end_epoch_time - start_time
       
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            print(f'New best loss is {best_loss}, model saved!')
        if dur > 432000: # After 5 days kill the job
            torch.save(model.state_dict(), save_path)
            print('THE TIME HAS EXCEEDED 5 DAYS, SAVING MODEL TO: ', save_path)
            break

    print('Training complete')

# function to get some data about our training epoch
def evaluate(model, loader, centers, device):
    model.eval()
    top1, top5, total = 0,0,0

    with torch.no_grad(): 
        for imgs, _, cell_ids in loader: # don't need soft labels, only images and ids
            imgs = imgs.to(device)
            cell_ids = cell_ids.to(device) 

            logits = model(imgs)
            probs = F.softmax(logits, dim=-1)

            top5_preds = probs.topk(5, dim=-1).indices
            top1_preds = top5_preds[:,0]

            top1 += (top1_preds == cell_ids).sum().item()
            top5 += (top5_preds == cell_ids.unsqueeze(1)).any(dim=1).sum().item()

            total += imgs.shape[0]

        eval_dict = {'top1': top1/total,
                     'top5': top5/total}
        return eval_dict

In [ ]:
# Our data is pretty uneven, so taking a simple 80/20 split may leave out too many of quads with only 50 images

def stratifiedSplit(samples, train_frac=0.8, seed=42):
    rng = random.Random(seed)
    by_cell = defaultdict(list)
    for path, cell_id in samples:
        by_cell[cell_id].append((path, cell_id))

    train_samples = []
    test_samples = []
    for cell_id, items in by_cell.items():
        rng.shuffle(items)
        split = max(1, int(train_frac * len(items)))
        train_samples.extend(items[:split])
        test_samples.extend(items[split:])

    rng.shuffle(train_samples)
    rng.shuffle(test_samples)
    return train_samples, test_samples

In [ ]:
BASE_PATH = './'
SAVE_PATH = './geoclip_us_states.pth'
lr = 3e-4
BATCH_SIZE = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GeoCLIPClassifier(num_classes=soft_labels.shape[0]).to(device)

random.shuffle(samples)
train_split = int(0.8*len(samples))

train_samples, val_samples = stratifiedSplit(samples)

train_data = GeoData(train_samples, soft_labels, aug=True)
val_data = GeoData(val_samples, soft_labels, aug=False)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
train(model, train_loader, val_loader)